# Create MITRE ATT&CK Tactics & Techniques json files

Download MITRE ATT&CK taxonomy data (most recent enterprise-attack, ics-attack & mobile-attack json files) from: https://github.com/mitre-attack/attack-stix-data

In [26]:
from mitreattack.stix20 import MitreAttackData
import json

In [30]:
def process_mitre_attack_json(json_file_path):
    """Process the ATT&CK data from a single ATT&CK json file."""
    
    # Initialize the MitreAttackData class
    mitre_attack_data = MitreAttackData(json_file_path)
    
    # Fetch all tactics
    tactics = mitre_attack_data.get_tactics_by_matrix()
    
    # Prepare the structure for the JSON
    tactics_list = []

    # We assume that our ATT&CK JSON files contain a single domain, which we extract here.
    if 'Enterprise ATT&CK' in list(tactics.keys()):
        domain_str = 'Enterprise ATT&CK'
        domain = 'enterprise-attack'
    # We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain.
    if 'Mobile ATT&CK' in list(tactics.keys()):
        domain_str = 'Mobile ATT&CK'
        domain = 'mobile-attack'
    if 'ATT&CK for ICS' in list(tactics.keys()):
        domain_str = 'ATT&CK for ICS'
        domain = 'ics-attack'
    
    # Iterate over each tactic
    for tactic in tactics[domain_str]:
        # Fetch all techniques for the current tactic
        techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], domain)
    
        # Prepare the techniques list with sub-techniques nested under their parent techniques
        techniques_dict = {}
        for technique in techniques:
            technique_entry = {
                "name": technique['name'],
                # "description": technique['description'],
                "external_id": technique['external_references'][0]['external_id'],
                "visibility": False  # Set default visibility to False, can be modified as needed
            }
            if technique.get('x_mitre_is_subtechnique'):
                parent_id = technique['external_references'][0]['external_id'].split('.')[0]
                if parent_id in techniques_dict:
                    if 'sub_techniques' not in techniques_dict[parent_id]:
                        techniques_dict[parent_id]['sub_techniques'] = []
                    techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
                else:
                    techniques_dict[parent_id] = {
                        "sub_techniques": [technique_entry]
                    }
            else:
                techniques_dict[technique_entry['external_id']] = technique_entry
    
        # Sort the techniques alphabetically by name
        techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])
    
        # Sort the sub-techniques alphabetically by name
        for technique in techniques_list:
            if 'sub_techniques' in technique:
                technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])
    
        # Prepare the tactic entry
        tactic_entry = {
            "name": tactic['name'],
            "external_id": tactic['external_references'][0]['external_id'],
            "techniques": techniques_list
        }
    
        # Add the tactic entry to the tactics list
        tactics_list.append(tactic_entry)

    return tactics_list

In [31]:
def process_all_mitre_attack_json_files():
    
    # Load JSON files of all ATT&CK domains.
    enterprise_tactics = process_mitre_attack_json("mitre_attack_files/enterprise-attack-15.1.json")
    ics_tactics = process_mitre_attack_json("mitre_attack_files/ics-attack-15.1.json")
    mobile_tactics = process_mitre_attack_json("mitre_attack_files/mobile-attack-15.1.json")
    
    # Prepare the final JSON structure.
    final_json = {
        "enterprise_tactics": enterprise_tactics,
        "ics_tactics": ics_tactics,
        "mobile_tactics": mobile_tactics
    }
    
    # Write the JSON to a file.
    with open("tactics_and_techniques.json", "w") as json_file:
        json.dump(final_json, json_file, indent=2)
    
    print("JSON file created successfully!")

In [32]:
process_all_mitre_attack_json_files()

JSON file created successfully!


In [24]:
# # Initialize the MitreAttackData class
# mitre_attack_data = MitreAttackData("mitre_attack_files/mobile-attack-15.1.json")

# # Fetch all tactics
# tactics = mitre_attack_data.get_tactics_by_matrix()

# # Prepare the structure for the JSON
# tactics_list = []

# list(tactics.keys())

In [4]:
# # Initialize the MitreAttackData class
# mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-15.1.json")

# # Fetch all tactics
# tactics = mitre_attack_data.get_tactics_by_matrix()

# # Prepare the structure for the JSON
# tactics_list = []

# # Iterate over each tactic
# for tactic in tactics['Enterprise ATT&CK']:
#     # Fetch all techniques for the current tactic
#     techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], "enterprise-attack")

#     # Prepare the techniques list with sub-techniques nested under their parent techniques
#     techniques_dict = {}
#     for technique in techniques:
#         technique_entry = {
#             "name": technique['name'],
#             # "description": technique['description'],
#             "external_id": technique['external_references'][0]['external_id'],
#             "visibility": False  # Set default visibility to False, can be modified as needed
#         }
#         if technique.get('x_mitre_is_subtechnique'):
#             parent_id = technique['external_references'][0]['external_id'].split('.')[0]
#             if parent_id in techniques_dict:
#                 if 'sub_techniques' not in techniques_dict[parent_id]:
#                     techniques_dict[parent_id]['sub_techniques'] = []
#                 techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
#             else:
#                 techniques_dict[parent_id] = {
#                     "sub_techniques": [technique_entry]
#                 }
#         else:
#             techniques_dict[technique_entry['external_id']] = technique_entry

#     # Sort the techniques alphabetically by name
#     techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])

#     # Sort the sub-techniques alphabetically by name
#     for technique in techniques_list:
#         if 'sub_techniques' in technique:
#             technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])

#     # Prepare the tactic entry
#     tactic_entry = {
#         "name": tactic['name'],
#         "external_id": tactic['external_references'][0]['external_id'],
#         "techniques": techniques_list
#     }

#     # Add the tactic entry to the tactics list
#     tactics_list.append(tactic_entry)

# # Prepare the final JSON structure
# final_json = {
#     "tactics": tactics_list
# }

# # Write the JSON to a file
# with open("tactics_and_techniques.json", "w") as json_file:
#     json.dump(final_json, json_file, indent=2)

# print("JSON file created successfully!")

JSON file created successfully!


# Data Exploration

In [91]:
tactics['Enterprise ATT&CK'][0]

{'x_mitre_domains': ['enterprise-attack'],
 'object_marking_refs': ['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'],
 'id': 'x-mitre-tactic--2558fd61-8c75-4730-94c4-11926db2a263',
 'type': 'x-mitre-tactic',
 'created': '2018-10-17T00:14:20.652Z',
 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5',
 'external_references': [{'external_id': 'TA0006',
   'url': 'https://attack.mitre.org/tactics/TA0006',
   'source_name': 'mitre-attack'}],
 'modified': '2022-04-25T14:00:00.188Z',
 'name': 'Credential Access',
 'description': 'The adversary is trying to steal account names and passwords.\n\nCredential Access consists of techniques for stealing credentials like account names and passwords. Techniques used to get credentials include keylogging or credential dumping. Using legitimate credentials can give adversaries access to systems, make them harder to detect, and provide the opportunity to create more accounts to help achieve their goals.',
 'x_mitre_version': '1

In [76]:
tactics['Enterprise ATT&CK'][0]['external_references'][0]['external_id']

'TA0043'

In [25]:
[print(x) for x in tactics['Enterprise ATT&CK'][0].keys()]

type
spec_version
id
created_by_ref
created
modified
revoked
external_references
object_marking_refs
name
description
x_mitre_attack_spec_version
x_mitre_domains
x_mitre_modified_by_ref
x_mitre_shortname
x_mitre_version


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [28]:
tactics['Enterprise ATT&CK'][0]

Tactic(type='x-mitre-tactic', spec_version='2.1', id='x-mitre-tactic--daa4cbb1-b4f4-4723-a824-7f1efd6e0592', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2020-10-02T14:48:41.809Z', modified='2022-04-25T14:00:00.188Z', revoked=False, external_references=[ExternalReference(source_name='mitre-attack', url='https://attack.mitre.org/tactics/TA0043', external_id='TA0043')], object_marking_refs=['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'], name='Reconnaissance', description='The adversary is trying to gather information they can use to plan future operations.\n\nReconnaissance consists of techniques that involve adversaries actively or passively gathering information that can be used to support targeting. Such information may include details of the victim organization, infrastructure, or staff/personnel. This information can be leveraged by the adversary to aid in other phases of the adversary lifecycle, such as using gathered information to plan a

In [35]:
[x for x in techniques[0].keys()]

['type',
 'spec_version',
 'id',
 'created_by_ref',
 'created',
 'modified',
 'name',
 'description',
 'kill_chain_phases',
 'revoked',
 'external_references',
 'object_marking_refs',
 'x_mitre_attack_spec_version',
 'x_mitre_data_sources',
 'x_mitre_detection',
 'x_mitre_domains',
 'x_mitre_is_subtechnique',
 'x_mitre_modified_by_ref',
 'x_mitre_platforms',
 'x_mitre_version']

In [42]:
techniques[0].x_mitre_is_subtechnique

False

In [43]:
[x.x_mitre_is_subtechnique for x in techniques]

[False,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 True,
 True,
 True,
 True,
 True]

In [58]:
[print(f"{x}: {techniques[0][x]}\n\n") for x in techniques[0]]

type: attack-pattern


spec_version: 2.1


id: attack-pattern--09312b1a-c3c6-4b45-9844-3ccc78e5d82f


created_by_ref: identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5


created: 2020-10-02 16:39:33.966000+00:00


modified: 2022-04-25 14:00:00.188000+00:00


name: Gather Victim Host Information


description: Adversaries may gather information about the victim's hosts that can be used during targeting. Information about hosts may include a variety of details, including administrative data (ex: name, assigned IP, functionality, etc.) as well as specifics regarding its configuration (ex: operating system, language, etc.).

Adversaries may gather this information in various ways, such as direct collection actions via [Active Scanning](https://attack.mitre.org/techniques/T1595) or [Phishing for Information](https://attack.mitre.org/techniques/T1598). Adversaries may also compromise sites then include malicious content designed to collect host information from visitors.(Citation: ATT ScanBox) 

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [71]:
techniques[0]['external_references'][0]['external_id']

'T1592'